# HanziGen - GBK 补字推理版（训练完成后的最终路线）

> 本版聚焦**最终路线**，适合已经完成 VQ-VAE / LDM 训练、只需补字生成的情况：
>
> ```
> Cell 0: 配置字体名 + 补字基准
> Cell 1: 环境初始化 / 断连自检（可重复运行，已就绪自动跳过）
> Cell 2: 生成 GBK 缺失字清单 + 改写 inference.sh
> Cell 3: 推理生成 + 评估指标（GPU）
> Cell 4: SVG 转换（纯 CPU，可切无卡实例）
> Cell 5: 下载 SVG zip → FontForge 导入原字体 → 导出完整字体
> ```

## 现在只需要执行哪些 Cell？

| 你的情况 | 需要运行的 Cell |
|---|---|
| **上次会话还在运行**（环境、仓库、checkpoints 都在） | **Cell 0 → Cell 2 → Cell 3 → Cell 4 → Cell 5** |
| **工作空间重新启动过** | **Cell 0 → Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5** |

> - Cell 1 是"自检型"的，已就绪的部分会自动跳过，任何时候都可以先跑一遍求个安心
> - Cell 2 是核心新增：以 **GBK 简体标准字符集（20,902 字）** 为基准，算出你的字体缺失的字并写入 `inference.sh`
> - Cell 3 / 4 / 5 就是原来的推理 + 转 SVG + 下载（无需重训模型）；Cell 4 可切无卡实例运行

---
## Cell 0: 配置参数

> **只改这里！** 填你放在 `fonts/` 目录下的字体文件名（英文）。

In [ ]:
# OpenMP 线程数：部分云镜像预置的 OMP_NUM_THREADS 值非法，会导致 libgomp 警告并按默认开满全部 CPU 核
# （须在 import torch/numpy 之前设置；注意用直接赋值强制覆盖——镜像里预置的值往往本身就是非法的，
#   setdefault 会因 键已存在 而跳过，导致非法值传给训练/推理子进程）
# 取值自动适配：实际可用核数的一半、封顶 8（大机器足够用，小机器避免与 DataLoader workers 抢核）
import os
try:
    _cpu = len(os.sched_getaffinity(0))   # Linux：本进程实际可用的核数
except AttributeError:
    _cpu = os.cpu_count() or 4            # Windows / macOS 回退
os.environ["OMP_NUM_THREADS"] = str(min(8, max(1, _cpu // 2)))

# ==================== 修改你的字体文件名 ====================
TARGET_FONT = "ChangguMingtiMedium.otf"   # 改成你放在 fonts/ 目录下的字体名
# ==========================================================

# ===== 补字基准选择（二选一）=====

# 注意：GB2312 的正确写法是 gb2312（不是 gbk2312）
# "gbk"    = GBK 简体标准字符集（20,902 字，简体+常用繁体，推荐）
# "gb2312" = 仅 GB2312 简体核心字（6,763 字，范围更小更保守）
CHARSET_BASE = "gbk"
# ==========================================================

FONT_NAME = TARGET_FONT.rsplit(".", 1)[0]
print(f"目标字体: {TARGET_FONT}")
print(f"字体名称: {FONT_NAME}")
print(f"补字基准: {CHARSET_BASE}")

# ==================== 参考字体设置 ====================
REFERENCE_FONTS_DIR = "fonts/jigmo"   # 参考字体目录（提供字形结构供模型生成）
REF_FONT_PRIORITY = ""                # 覆盖优先级：逗号分隔文件名，空=按文件名顺序（jigmo→jigmo2→jigmo3）
REF_FONT_MODE = "first"               # first=优先字体先命中（推荐）/ last=旧行为（后写覆盖）
REF_FONT_STRICT = False               # 严格模式：True=只使用优先级列表中的字体（风格彻底统一）；
                                      #   新参考未 100% 覆盖缺字表时会导致推理报错，一般保持 False

import re as _re
os.environ["HANZIGEN_REF_FONT_PRIORITY"] = REF_FONT_PRIORITY
os.environ["HANZIGEN_REF_FONT_MODE"] = REF_FONT_MODE
os.environ["HANZIGEN_REF_FONT_STRICT"] = "1" if REF_FONT_STRICT else "0"
print(f"参考字体: {REFERENCE_FONTS_DIR} | 优先级: {REF_FONT_PRIORITY or '默认文件名顺序'} | 模式: {REF_FONT_MODE}")
# （REFERENCE_FONTS_DIR 会在 Cell 2 改写进 inference.sh）

# ==================== 推理批大小（Cell 3 推理阶段生效）====================
# LDM 在 latent 空间（64x64）采样，显存占用极低；调大可提高 GPU 利用率且不损失精度
# （每个字独立采样，batch 大小不影响生成质量，只影响速度）
#   64=默认(16G 显存稳妥)  128=显存充裕(24G+)想更快  32=显存较小/OOM 时
INFERENCE_BATCH_SIZE = 64
# =========================================================================

---
## Cell 1: 环境初始化 + 断连自检（可重复运行）

> 检查/克隆仓库 → 安装依赖 → 检查字体 → 本地 Jigmo*.zip 优先 / 下载 Jigmo → 改写脚本 → 验证 GPU。
> **已就绪的部分会自动跳过**，断连重启后重跑本 Cell 即可恢复环境。

In [ ]:
import os, shutil, json, re, sys, glob, zipfile, io, urllib.request, subprocess
from fontTools.ttLib import TTFont

# ===== 0. 克隆项目代码（首次运行时自动拉取）=====
REPO_URL  = "https://github.com/ICW-k/HanziGen_ICWfork.git"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")

is_project_root = os.path.isdir("scripts") and os.path.exists("requirements.txt")

if not is_project_root:
    found_repo = None
    for search_root in [os.getcwd(), "/workspace", "/home", "/"]:
        if not os.path.isdir(search_root):
            continue
        try:
            for entry in os.listdir(search_root):
                candidate = os.path.join(search_root, entry)
                if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, "scripts")):
                    if os.path.exists(os.path.join(candidate, "requirements.txt")):
                        found_repo = candidate
                        break
        except PermissionError:
            continue
        if found_repo:
            break

    if found_repo:
        print(f"发现已有项目目录: {found_repo}")
        os.chdir(found_repo)
    else:
        print(f"正在克隆仓库: {REPO_URL}")
        subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir(REPO_NAME)

    print(f"已切换到项目根目录: {os.getcwd()}")
else:
    print("已在项目根目录，跳过克隆")

# Cloud Studio 兼容：确保 python 命令指向 python3
print("\n===== 环境兼容检查 =====")
!which python3 && ln -sf $(which python3) /usr/local/bin/python 2>/dev/null; python --version && echo "python 命令已就绪"

# ===== 1. 确认工作目录 =====
PROJECT = os.getcwd()
print(f"\n工作目录: {PROJECT}")
!ls -F | head -30

# ===== 2. 安装依赖 =====
print("\n===== 安装 PyTorch + 依赖 =====")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q -r requirements.txt
print("\n依赖安装完成！")
# ===== 依赖自检：补齐 pip 未覆盖 / 编译失败的包（如 C 扩展 potracer）=====
import importlib, sys, subprocess, shutil, os
REQUIRED_PACKAGES = {
    "torch": "torch", "torchvision": "torchvision", "rich": "rich",
    "fontTools": "fonttools", "lpips": "lpips", "matplotlib": "matplotlib",
    "skimage": "scikit-image", "cleanfid": "clean-fid",
    "potrace": "potracer",          # pip 包名 potracer，import 名 potrace
    "tqdm": "tqdm", "einops": "einops", "PIL": "pillow", "numpy": "numpy",
}
NO_AUTO_INSTALL = {"torch", "torchvision"}   # 装错版本代价大，只提示

def _can_import(mod):
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False

missing = [(m, p) for m, p in REQUIRED_PACKAGES.items() if not _can_import(m)]
if not missing:
    print("  依赖自检通过")
else:
    print(f"  缺失模块: {[m for m, _ in missing]}")
    to_install = [p for m, p in missing if m not in NO_AUTO_INSTALL]
    if to_install:
        print(f"  正在补装: {to_install}")
        r = subprocess.run([sys.executable, "-m", "pip", "install", *to_install],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print("  [ERROR] pip 安装失败，输出尾部：")
            print("\n".join((r.stdout + r.stderr).strip().splitlines()[-15:]))
    still = [m for m in REQUIRED_PACKAGES if not _can_import(m)]
    if still:
        print(f"  [WARN] 仍未就绪: {still}")
        if "torch" in still:
            print("        请手动安装与 CUDA 匹配的 torch，例如：")
            print("        pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124")
        if "potrace" in still:
            print("        potracer 是 C 扩展包，编译失败时先装系统库：")
            print("          apt-get update && apt-get install -y build-essential libpotrace-dev")
            print("        缺失只影响 SVG 转换（Cell 6 / Cell 5 后半），训练不受影响")

_sh_py = shutil.which("python") or shutil.which("python3")
if _sh_py and os.path.abspath(_sh_py) != os.path.abspath(sys.executable):
    print(f"  [WARN] 终端 python 与 notebook kernel 不一致：\n        {_sh_py}\n        {sys.executable}")
    print("        scripts/*.sh 用终端 python 执行，包需装到该解释器，或统一到同一环境后重启 kernel")
else:
    print(f"  python 解释器一致: {_sh_py}")

# ===== 3. 检查目标字体（智能定位）=====
print("\n===== 检查字体 =====")

def find_font_in_workspace(filename: str, search_root: str = ".") -> str | None:
    """在 search_root 下递归搜索 filename，返回第一个匹配的路径"""
    for root, dirs, files in os.walk(search_root, followlinks=False):
        dirs[:] = [d for d in dirs if not d.startswith(".") and d not in ("__pycache__", "node_modules")]
        for f in files:
            if f.lower() == filename.lower():
                return os.path.join(root, f)
    return None

font_path = f"fonts/{TARGET_FONT}"

print("当前 fonts/ 目录内容:")
if os.path.isdir("fonts"):
    !find fonts/ -type f 2>/dev/null | head -30
else:
    print("  [WARN] fonts/ 目录不存在！")
    parent_fonts = os.path.join(os.path.dirname(PROJECT) if os.path.dirname(PROJECT) != PROJECT else "..", "fonts")
    if os.path.isdir(parent_fonts):
        print(f"  但发现上级目录有: {parent_fonts}/")

if not os.path.exists(font_path):
    print(f"\n[WARN] {font_path} 不存在，正在搜索整个工作区...")
    found = find_font_in_workspace(TARGET_FONT, "/workspace" if os.path.isdir("/workspace") else ".")
    if found:
        print(f"[OK] 找到字体: {found}")
        os.makedirs("fonts", exist_ok=True)
        shutil.copy2(found, font_path)
        print(f"[OK] 已复制到: {font_path}")
    else:
        !ls -la
        raise FileNotFoundError(
            f"\n字体文件 '{TARGET_FONT}' 在整个工作区都找不到！\n"
            f"\n请检查:\n"
            f"  1. 文件名是否完全一致（注意大小写）? 当前配置: '{TARGET_FONT}'\n"
            f"  2. 字体是否已上传到 Cloud Studio 工作空间?\n"
            f"  3. 上传后文件是否放在 fonts/ 子目录下?"
        )
else:
    print(f"目标字体已就绪: {font_path}")

# ===== 4. 把字体路径写进所有 .sh 脚本 =====
print("\n===== 改写脚本字体路径 =====")
for sh_file in glob.glob("scripts/*.sh"):
    with open(sh_file, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r'fonts/[^\x22\x27]+?\.(?:ttf|otf|TTF|OTF)', f'fonts/{TARGET_FONT}', content)
    with open(sh_file, "w", encoding="utf-8") as f:
        f.write(content)
print("脚本字体路径已统一替换")

# ===== 5. Jigmo 参考字体：从官方 ZIP 下载 =====
print("\n===== 准备 Jigmo 参考字体 =====")

jigmo_files = ["jigmo.ttf", "jigmo2.ttf", "jigmo3.ttf"]

def find_jigmo_zip(roots):
    """在工作区递归搜索 Jigmo*.zip（文件名含 Jigmo 标识即可，版本不限）"""
    for search_root in roots:
        if not os.path.isdir(search_root):
            continue
        for root, dirs, files in os.walk(search_root, followlinks=False):
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in ("__pycache__", "node_modules")]
            for f in files:
                if f.lower().endswith(".zip") and "jigmo" in f.lower():
                    return os.path.join(root, f)
    return None

def extract_jigmo_from_zip(zip_path, target_dir="fonts/jigmo"):
    """从 Jigmo ZIP 中解压 jigmo.ttf / jigmo2.ttf / jigmo3.ttf"""
    os.makedirs(target_dir, exist_ok=True)
    zf = zipfile.ZipFile(zip_path)
    for zip_name in zf.namelist():
        basename = os.path.basename(zip_name).lower()
        if basename in jigmo_files:
            zf.extract(zip_name, target_dir)
            extracted = os.path.join(target_dir, zip_name)
            target = os.path.join(target_dir, basename)
            if extracted != target:
                if os.path.exists(target):
                    os.remove(target)
                os.rename(extracted, target)
    zf.close()
    return True

def download_jigmo_fonts(target_dir="fonts/jigmo"):
    os.makedirs(target_dir, exist_ok=True)
    zip_url = "https://kamichikoichi.github.io/jigmo/Jigmo-20250912.zip"
    print(f"  下载 Jigmo ZIP: {zip_url}")
    try:
        resp = urllib.request.urlopen(zip_url, timeout=30)
        data = resp.read()
        if len(data) < 10000:
            raise ValueError(f"下载数据太小 ({len(data)} bytes)")
        zf = zipfile.ZipFile(io.BytesIO(data))
    except Exception as e:
        print(f"  [ERROR] ZIP 下载失败: {e}")
        return False
    for zip_name in zf.namelist():
        basename = os.path.basename(zip_name).lower()
        if basename in jigmo_files:
            zf.extract(zip_name, target_dir)
            extracted = os.path.join(target_dir, zip_name)
            target = os.path.join(target_dir, basename)
            if extracted != target:
                if os.path.exists(target):
                    os.remove(target)
                os.rename(extracted, target)
    zf.close()
    return True

def validate_font_file(fpath):
    try:
        f = TTFont(fpath)
        if "cmap" not in f:
            return False, "缺少 cmap 表"
        return True, f"OK ({len(f.getBestCmap())} glyphs)"
    except Exception as e:
        return False, str(e)[:80]

need_download = False
for fname in jigmo_files:
    fpath = f"fonts/jigmo/{fname}"
    if os.path.exists(fpath):
        valid, msg = validate_font_file(fpath)
        if not valid:
            print(f"  [WARN] {fname} 无效 ({msg})")
            os.remove(fpath)
            need_download = True
    else:
        need_download = True

if need_download:
    # ---- 本地 Jigmo ZIP 优先：先搜工作区已有的 Jigmo*.zip（版本不限），找不到才在线下载 ----
    jigmo_zip = find_jigmo_zip([PROJECT, "/workspace", "/home", "/root"])
    if jigmo_zip:
        print(f"  [OK] 找到本地 Jigmo ZIP: {jigmo_zip}，直接解压使用")
        extract_jigmo_from_zip(jigmo_zip)
        still_missing = [f for f in jigmo_files if not os.path.exists(f"fonts/jigmo/{f}")]
        if not still_missing:
            need_download = False
            print("  [OK] 已从本地 ZIP 提取全部 Jigmo 字体")
        else:
            print(f"  [WARN] 本地 ZIP 缺少: {still_missing}，将下载官方版本补齐")
    if not download_jigmo_fonts():
        raise RuntimeError("Jigmo 下载失败！")
    print("Jigmo 字体下载完成，验证中...")
    for fname in jigmo_files:
        valid, msg = validate_font_file(f"fonts/jigmo/{fname}")
        print(f"    [{'OK' if valid else 'ERROR'}] {fname}: {msg}")
        if not valid:
            raise RuntimeError(f"Jigmo 字体 {fname} 验证失败！")
else:
    print("Jigmo 字体已就绪，跳过下载")

# ===== 6. 断连自检 =====
print("\n===== 断连自检 =====")
os.makedirs("checkpoints", exist_ok=True)

vqvae_ckpt = f"checkpoints/vqvae_{FONT_NAME}.pth"
ldm_ckpt = f"checkpoints/ldm_{FONT_NAME}.pth"
print(f"  VQ-VAE 检查点:   {'存在: '+vqvae_ckpt if os.path.exists(vqvae_ckpt) else '不存在'}")
print(f"  LDM 检查点:      {'存在: '+ldm_ckpt if os.path.exists(ldm_ckpt) else '不存在'}")
if not os.path.exists(ldm_ckpt):
    print("\n  [WARN] LDM 检查点不存在！若你尚未完成训练，请回到完整版 notebook 完成 Cell 2-4。")

# ===== 7. 验证 GPU =====
print("\n===== 验证 GPU =====")
import torch
assert torch.cuda.is_available(), "GPU 不可用！请在 Cloud Studio 工作空间设置中切换到 GPU 实例"
prop = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"显存: {prop.total_memory / 1024**3:.1f} GB")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")
print(f"\n全部初始化完成！字体: {font_path}")

---
## Cell 2: 生成补字字符集（GBK 缺失字）+ 改写 inference.sh

> 核心步骤：以 **GBK（20,902 字）** 为基准，算出你的字体缺失的字，写入字符集文件，并让 `inference.sh` 指向它。
>
> - 生成结果 = GBK 汉字 − 你的字体已覆盖的字（简体为主）
> - 字体已有的字**不会被重新生成**（保留原字形，质量无损）
> - 若想只补简体核心字，在 Cell 0 把 `CHARSET_BASE` 改为 `"gb2312"`

In [ ]:
import os, re
from fontTools.ttLib import TTFont

font_path = f"fonts/{TARGET_FONT}"

# ===== 1. 校验并构建基准字符集（GBK 或 GB2312，系统自带编码器，无需额外文件）=====
_CHARSET_ALIAS = {
    "gbk": "gbk",
    "gb2312": "gb2312",
    "gbk2312": "gb2312",   # 兼容手误（GB2312 的正确写法是 gb2312）
    "GBK": "gbk",
    "GB2312": "gb2312",
}
if CHARSET_BASE not in _CHARSET_ALIAS:
    raise ValueError(
        f"CHARSET_BASE 取值错误: {CHARSET_BASE!r}\n"
        f"可选值只有: \"gbk\"（推荐）或 \"gb2312\"\n"
        f"提示：GB2312 的正确写法是 gb2312，不是 gbk2312")
enc = _CHARSET_ALIAS[CHARSET_BASE]
CHARSET_BASE = enc   # 归一化，避免别名/大小写影响后续目录名
base_chars = set()
for cp in range(0x4E00, 0xA000):          # CJK 基本区全部码位
    try:
        chr(cp).encode(enc)                # 能按该编码集编码 = 属于该字符集
        base_chars.add(chr(cp))
    except UnicodeEncodeError:
        pass
print(f"[{CHARSET_BASE}] 基准字符集: {len(base_chars)} 字")

# ===== 2. 读字体已覆盖的码位 =====
font = TTFont(font_path, fontNumber=0)
cmap = set()
for table in font["cmap"].tables:
    if table.isUnicode():
        cmap.update(table.cmap.keys())
print(f"字体总码位: {len(cmap)}")

# ===== 3. 缺失 = 基准字符集 - 字体已有 =====
missing = sorted(base_chars - {chr(c) for c in cmap})
print(f"{CHARSET_BASE} 中字体缺失: {len(missing)} 字")
print("前 20 个缺失字:", "".join(missing[:20]))

# ===== 4. 写入自定义字符集 =====
out_dir = f"charsets/{CHARSET_BASE}_coverage/{FONT_NAME}"
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/missing.txt"
with open(out_path, "w", encoding="utf-8") as f:
    f.write("\n".join(missing))
print(f"[OK] 已写入 {out_path}")

# ===== 5. 改写 inference.sh：指向新字符集 + 切回 GPU =====
with open("scripts/inference.sh", encoding="utf-8") as f:
    content = f.read()
content = re.sub(r'CHARSET_PATH="[^"]*"', f'CHARSET_PATH="{out_path}"', content)
content = re.sub(r'DEVICE="[^"]*"', 'DEVICE="cuda"', content)
content = re.sub(r'REFERENCE_FONTS_DIR="[^"]*"', f'REFERENCE_FONTS_DIR="{REFERENCE_FONTS_DIR}"', content)
content = re.sub(r'BATCH_SIZE=\d+', f'BATCH_SIZE={INFERENCE_BATCH_SIZE}', content)
print(f"[OK] 参考字体目录: {REFERENCE_FONTS_DIR} | 优先级: {REF_FONT_PRIORITY or '默认文件名顺序'} | 模式: {REF_FONT_MODE}")
with open("scripts/inference.sh", "w", encoding="utf-8") as f:
    f.write(content)
print("[OK] inference.sh 已指向新字符集并切回 GPU")
print("\n下一步：运行 Cell 3 开始推理生成（13,000+ 字建议预留 30-60 分钟 GPU 机时）")

---
## Cell 3: 推理生成 + 评估指标（GPU）

> 依赖 LDM 训练完成（`checkpoints/ldm_{FONT_NAME}.pth` 存在）。
> 生成范围 = Cell 2 写入的 GBK 缺失字清单。
>
> SVG 转换已拆分到 **Cell 4**（纯 CPU）——本 Cell 结束后可切无卡实例再运行 Cell 4。

In [ ]:
import os, subprocess

ldm_ckpt = f"checkpoints/ldm_{FONT_NAME}.pth"

if not os.path.exists(ldm_ckpt):
    print(f"{ldm_ckpt} 不存在，请先完成模型训练（完整版 notebook Cell 2-4）")
else:
    print("\n===== 推理生成 =====")
    subprocess.run(["bash", "scripts/inference.sh"], check=True)
    print("\n===== 计算评估指标 =====")
    try:
        subprocess.run(["bash", "scripts/compute_metrics.sh"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"[WARN] 评估指标计算失败（返回码 {e.returncode}），但不影响补字结果。")
        print("       常见原因: cleanfid 的 InceptionV3 模型下载不完整/损坏。")
        print("       修复方式: rm -f /tmp/torchscript_inception.pth && rm -rf ~/.cache/cleanfid 后重跑本 Cell")
    print("\n===== GPU 阶段完成！SVG 转换请运行 Cell 4（纯 CPU，可切无卡实例） =====")
    print(f"生成的样本位置: samples_{FONT_NAME}/")

---
## Cell 4: SVG 转换（纯 CPU，可在无卡实例运行）

> 依赖 Cell 3 产物 `samples_{FONT_NAME}/inference/gen/`。与 GPU 无关——推理完成后切无卡实例再运行本 Cell，省 GPU 机时。
>
> 转换完成后运行 Cell 5 导出 zip。

In [ ]:
import os, re, subprocess

# ===== CPU 核数检测与转换并行度分配 =====
try:
    cpu_cores = len(os.sched_getaffinity(0))   # Linux：本进程实际可用的核数
except AttributeError:
    cpu_cores = os.cpu_count() or 4            # Windows / macOS 回退
convert_workers = max(1, min(32, cpu_cores - 2))
print(f"检测到可用 CPU 核数: {cpu_cores} → SVG 转换进程数: {convert_workers}")

with open("scripts/convert_to_svg.sh", encoding="utf-8") as f:
    content = f.read()
content = re.sub(r"NUM_WORKERS=auto", f"NUM_WORKERS={convert_workers}", content)
with open("scripts/convert_to_svg.sh", "w", encoding="utf-8") as f:
    f.write(content)

png_dir = f"samples_{FONT_NAME}/inference/gen"
svg_dir = f"svgs_{FONT_NAME}"

if not os.path.isdir(png_dir):
    print(f"{png_dir} 不存在，请先在 GPU 实例完成 Cell 3 推理")
else:
    n_png = len([f for f in os.listdir(png_dir) if f.endswith(".png")])
    print(f"待转换 PNG: {n_png} 张（{png_dir}/）")
    print("\n===== SVG 转换（多进程并行，CPU 跑满后上万张约数分钟~数十分钟） =====")
    subprocess.run(["bash", "scripts/convert_to_svg.sh"], check=True)

    n_svg = len([f for f in os.listdir(svg_dir) if f.endswith(".svg")]) if os.path.isdir(svg_dir) else 0
    print("\n===== 转换完成 =====")
    print(f"SVG 输出: {n_svg} 个 → {svg_dir}/")
    print("下一步：运行 Cell 5 导出 zip")

---
## Cell 5: 导出并下载 SVG 结果

> 把 `svgs_{FONT_NAME}/` 打包成 zip → 生成浏览器下载链接 + 同步保存到工作空间根目录（文件树可见处）+ 页面内预览前 9 个。
>
> 下载后即可用 **FontForge**：打开原字体 → Import 这些 SVG → Generate 导出完整字体。

In [ ]:
import os, glob, zipfile, io, base64
from IPython.display import HTML, display

svg_dir = f"svgs_{FONT_NAME}"
if not os.path.isdir(svg_dir):
    print(f"{svg_dir} 不存在，请先运行 Cell 4 完成 SVG 转换")
else:
    svg_files = sorted(glob.glob(os.path.join(svg_dir, "*.svg")))
    print(f"共找到 {len(svg_files)} 个 SVG 文件")

    # 1. 打包成 zip（内存中生成）
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in svg_files:
            zf.write(f, arcname=os.path.basename(f))
    zip_data = zip_buffer.getvalue()
    zip_name = f"svgs_{FONT_NAME}.zip"

    # 2. 保存到项目根目录（git clone 后的根目录）
    saved_paths = []
    roots = [os.path.abspath(os.getcwd())]   # 仅项目根，不散落到 /home、/root 等处
    seen = set()
    for r in roots:
        if r and r not in seen and os.path.isdir(r):
            seen.add(r)
            try:
                p = os.path.join(r, zip_name)
                with open(p, "wb") as f:
                    f.write(zip_data)
                saved_paths.append(p)
            except Exception:
                pass
    for p in saved_paths:
        print(f"  [已保存] {p} ({len(zip_data)/1024:.0f} KB)")

    # 3. 下载：仅小 zip 用浏览器 base64 直链；大 zip 嵌入 HTML 会卡死内核（挂起不释放）
    if len(zip_data) < 10 * 1024 * 1024:
        b64 = base64.b64encode(zip_data).decode()
        download_link = (
            '<a href="data:application/zip;base64,' + b64 + f'" download="{zip_name}" '
            'style="display:inline-block;font-size:18px;font-weight:bold;color:#fff;'
            'background:#1a73e8;padding:12px 28px;border-radius:8px;'
            'text-decoration:none;">'
            f'下载全部 SVG ({len(svg_files)} 个 / {len(zip_data)/1024:.0f} KB)</a>'
        )
        display(HTML(download_link))
    else:
        print(f"[提示] zip 较大（{len(zip_data)/1024/1024:.0f} MB），浏览器 base64 直链会卡死内核，已跳过")
        print(f"       请从左侧文件树找到项目根目录下的 {zip_name} 右键下载")

    # 4. 页面内预览前 9 个 SVG
    cards = []
    for f in svg_files[:9]:
        with open(f, encoding="utf-8") as fh:
            svg = fh.read()
        svg = svg.replace("<svg ", '<svg width="80" height="80" style="background:#fff;" ', 1)
        name = os.path.splitext(os.path.basename(f))[0]
        cards.append(
            f'<div style="border:1px solid #e0e0e0;border-radius:8px;padding:8px;'
            f'text-align:center;width:96px;">{svg}<div style="font-size:12px;color:#555;">{name}</div></div>'
        )
    display(HTML('<div style="display:flex;flex-wrap:wrap;gap:10px;">' + "".join(cards) + "</div>"))

---
## 断连恢复指南

工作空间重新启动后，按顺序运行：**Cell 0 → Cell 1 → Cell 2 → Cell 3 → Cell 4**。

- Cell 1 会自检环境、自动补齐缺失的依赖 / 本地 Jigmo*.zip 优先 / 字体路径
- Cell 2 会重新生成字符集清单（幂等，可重复运行）
- Cell 3 生成的 PNG 会**覆盖同名文件**，中断后重跑即可，不会混入旧数据

---

## 常见问题

| 问题 | 解决 |
|------|------|
| GPU 不可用 | Cloud Studio 工作空间设置中切换到 GPU 运行时 |
| LDM 检查点不存在 | 说明还没训练完，需回到完整版 notebook 完成训练 |
| 生成太慢 | 可把 `scripts/inference.sh` 的 `SAMPLE_STEPS=50` 改为 `20`（速度翻倍，质量略降） |
| 想只补简体核心字 | Cell 0 把 `CHARSET_BASE` 改为 `"gb2312"` 再重跑 Cell 2 |
| 想扩大范围到生僻字 | 把 Cell 2 的基准换成 unihan basic + ext_a（见对话方案） |